# Feature Construction Challenge -- Telco Churn
## Module 3, Class 3

**Objective:** Build new features from existing columns and measure their impact on model performance.

### What you will practice
- Creating interaction and ratio features
- Counting categorical flags
- Measuring feature impact with before/after comparison

---

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

print("Setup complete.")

## 1. Load and Clean Data

In [ ]:
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

# Fix TotalCharges
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

# Standardize verbose 'No X service' values
replace_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']
for col in replace_cols:
    df[col] = df[col].replace('No internet service', 'No')
df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No')

# Encode target
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

# Drop customerID
df = df.drop('customerID', axis=1)

print(f"Shape: {df.shape}")
print(f"Churn distribution:\n{df['Churn'].value_counts()}")
df.head()

## 2. Baseline Model (No Engineered Features)

We train a LogisticRegression on just the original numeric + encoded categorical features to establish a baseline F1 score.

In [ ]:
# Prepare baseline features: one-hot encode categoricals
cat_cols = df.select_dtypes(include='object').columns.tolist()
df_baseline = pd.get_dummies(df, columns=cat_cols, drop_first=True)

X_base = df_baseline.drop('Churn', axis=1)
y = df_baseline['Churn']

X_train_b, X_test_b, y_train, y_test = train_test_split(
    X_base, y, test_size=0.2, random_state=42, stratify=y
)

scaler_b = StandardScaler()
X_train_bs = scaler_b.fit_transform(X_train_b)
X_test_bs = scaler_b.transform(X_test_b)

model_base = LogisticRegression(max_iter=1000, random_state=42)
model_base.fit(X_train_bs, y_train)
y_pred_base = model_base.predict(X_test_bs)

f1_base = f1_score(y_test, y_pred_base)
print(f"Baseline F1 (class 1): {f1_base:.4f}")
print()
print(classification_report(y_test, y_pred_base))

---
## 3. Feature Construction

We create 5 new features that capture relationships the model can't learn from raw columns alone.

In [ ]:
df_eng = df.copy()

# Feature 1: Average charge per month of tenure
# Customers with high total charges but low tenure are paying more per month
df_eng['charge_per_month'] = df_eng['TotalCharges'] / df_eng['tenure'].replace(0, 1)
print(f"charge_per_month -- min: {df_eng['charge_per_month'].min():.2f}, max: {df_eng['charge_per_month'].max():.2f}")

In [ ]:
# Feature 2: Interaction between MonthlyCharges and tenure
# Captures the combined effect: long tenure + high charges = different risk profile
df_eng['monthly_charge_x_tenure'] = df_eng['MonthlyCharges'] * df_eng['tenure']
print(f"monthly_charge_x_tenure -- min: {df_eng['monthly_charge_x_tenure'].min():.2f}, max: {df_eng['monthly_charge_x_tenure'].max():.2f}")

In [ ]:
# Feature 3: Number of services subscribed
# More services = more engaged (potentially less likely to churn)
service_cols = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
                'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

df_eng['num_services'] = df_eng[service_cols].apply(
    lambda row: (row == 'Yes').sum(), axis=1
)
print(f"num_services distribution:\n{df_eng['num_services'].value_counts().sort_index()}")

In [ ]:
# Feature 4: Tenure in years (easier to interpret, captures non-linearity)
df_eng['tenure_years'] = df_eng['tenure'] / 12
print(f"tenure_years -- min: {df_eng['tenure_years'].min():.2f}, max: {df_eng['tenure_years'].max():.2f}")

In [ ]:
# Feature 5: Charge per service
# How much each service costs on average. High = expensive per service.
df_eng['charge_per_service'] = df_eng['MonthlyCharges'] / df_eng['num_services'].replace(0, 1)
print(f"charge_per_service -- min: {df_eng['charge_per_service'].min():.2f}, max: {df_eng['charge_per_service'].max():.2f}")

In [ ]:
# Show all new features
new_feat_cols = ['charge_per_month', 'monthly_charge_x_tenure', 'num_services',
                 'tenure_years', 'charge_per_service']
df_eng[new_feat_cols].describe().round(2)

---
## 4. Model WITH Engineered Features

In [ ]:
# Prepare features: one-hot encode categoricals on the engineered dataframe
cat_cols_eng = df_eng.select_dtypes(include='object').columns.tolist()
df_eng_encoded = pd.get_dummies(df_eng, columns=cat_cols_eng, drop_first=True)

X_eng = df_eng_encoded.drop('Churn', axis=1)
y_eng = df_eng_encoded['Churn']

X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(
    X_eng, y_eng, test_size=0.2, random_state=42, stratify=y_eng
)

scaler_e = StandardScaler()
X_train_es = scaler_e.fit_transform(X_train_e)
X_test_es = scaler_e.transform(X_test_e)

model_eng = LogisticRegression(max_iter=1000, random_state=42)
model_eng.fit(X_train_es, y_train_e)
y_pred_eng = model_eng.predict(X_test_es)

f1_eng = f1_score(y_test_e, y_pred_eng)
print(f"Engineered F1 (class 1): {f1_eng:.4f}")
print()
print(classification_report(y_test_e, y_pred_eng))

## 5. Comparison

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Baseline (original features)', 'With engineered features'],
    'Num Features': [X_base.shape[1], X_eng.shape[1]],
    'F1 Score (Churn=1)': [f1_base, f1_eng]
})
comparison['F1 Score (Churn=1)'] = comparison['F1 Score (Churn=1)'].round(4)
comparison['Improvement'] = ['--', f"{(f1_eng - f1_base) / f1_base * 100:+.1f}%"]
comparison

---
## TODO: Create 2 More Features

Think about what other combinations might be useful. Ideas:
- `has_partner_and_dependents`: 1 if both Partner=Yes AND Dependents=Yes, else 0
- `tenure_bucket`: map tenure into bins (0-12, 13-24, 25-48, 49-72)
- `overpaying`: MonthlyCharges > median MonthlyCharges for their contract type
- `no_protection`: 1 if no OnlineSecurity AND no OnlineBackup AND no DeviceProtection

1. Create 2 new features
2. Retrain the model with all features (original + 5 from above + your 2)
3. Report F1 and compare to both baseline and the 5-feature model

In [ ]:
# TODO: Create your 2 new features on df_eng
# df_eng['your_feature_1'] = ...
# df_eng['your_feature_2'] = ...

# Your code here


In [ ]:
# TODO: Retrain and report F1
# Follow the same pattern as Section 4:
#   1. One-hot encode
#   2. Train/test split (same random_state=42)
#   3. Scale
#   4. Train LogisticRegression
#   5. Compute F1

# Your code here


In [ ]:
# TODO: Build final comparison table with all 3 rows
# Baseline | 5-feature model | Your 7-feature model

# Your code here


---
## Summary

| Feature type | Example | Why it helps |
|-------------|---------|-------------|
| Ratio | charge_per_month | Captures rate, not just absolute value |
| Interaction | monthly_charge_x_tenure | Captures combined effect of two features |
| Count/Aggregate | num_services | Summarizes multiple columns into one signal |
| Transform | tenure_years | Changes scale, can help linear models |
| Derived ratio | charge_per_service | Normalizes one feature by another |